[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/cadquery_basics/blob/master/docs/tuto_colab/cadquery_multiplos_pav_perfil.ipynb)




# Exercício: Multiplos Pavimentos com importação de perfis
## CadQuery para Arquitetos e Engenheiros
### Versão Google Colab

Importando perfis do Autocad(dxf) e do rhino(step)



## Instalação

In [17]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("Running in Colab, installing packages...")
    !pip install cadquery cadquery-simpleviewer
else:
    print("Not running in Colab, skipping package installation.")

Not running in Colab, skipping package installation.


## Importação dos pacotes

In [18]:
import cadquery as cq
from cadquery_simpleviewer.viewer import show

## Implementação

In [19]:
# ── Parâmetros ──────────────────────────────────────────────────────────────
cota_inicial = 0
pap     = 3
n_pav   = 30


# ── Leitura do perfil ────────────────────────────────────────────────────────
perfil = cq.importers.importDXF("perfil_autocad.dxf", include=["profile_01"])
# perfil = cq.importers.importStep("perfil_step_214.step").val().scale(0.001)           # STEP from Rhino

# ── Geração dos pavimentos ───────────────────────────────────────────────────
lista_pav = []

for i in range(n_pav + 1):
    cota_atual = round(cota_inicial + pap * i, 2)

    pav = (
        cq.Workplane()
        .add(perfil)
        .wires().toPending()
        .extrude(pap)
        .translate((0, 0, cota_atual))

    )

    lista_pav.append(pav)
    # print(f"Cota do pavimento {i} = {cota_atual}")

show(lista_pav)



### Rotação incremental
copie o código acima e
adicione uma rotação gradual a cada andar

In [20]:
# ── Parâmetros ──────────────────────────────────────────────────────────────
cota_inicial = 0
pap     = 3
n_pav   = 30
rot_inc = 2.5

# ── Leitura do perfil ────────────────────────────────────────────────────────
perfil = cq.importers.importDXF("perfil_autocad.dxf", include=["profile_01"])
# perfil = cq.importers.importStep("perfil_step_214.step").val().scale(0.001)           # STEP from Rhino

# ── Geração dos pavimentos ───────────────────────────────────────────────────
lista_pav = []



for i in range(n_pav + 1):
    cota_atual = round(cota_inicial + pap * i, 2)

    pav = (
        cq.Workplane()
        .add(perfil)
        .wires().toPending()
        .extrude(pap)
        .translate((0, 0, cota_atual))
        .rotate((0, 0, 0), (0, 0, 1), i * rot_inc)

    )

    lista_pav.append(pav)
    # print(f"Cota do pavimento {i} = {cota_atual}")

show(lista_pav)




In [21]:
import numpy as np

def scale_xy(shape, scale =[1,1,1], translate=[0,0,0]):
    """Scales X and Y and translates Z in a single matrix operation."""
    m = cq.Matrix([
        [scale[0],  0,        0,        translate[0]],
        [0,         scale[1], 0,        translate[1]],
        [0,         0,        scale[2], translate[2]],
        [0,         0,        0,        1           ]
    ])
    return shape.transformGeometry(m)

# ── Parâmetros ──────────────────────────────────────────────────────────────
cota_inicial = 0
pap     = 3
n_pav   = 30
rot_inc = 2.5

smooth_factor = 1/(np.pi *1.5)

# ── Leitura do perfil ────────────────────────────────────────────────────────
perfil = cq.importers.importDXF("perfil_autocad.dxf", include=["profile_01"])
# perfil = cq.importers.importStep("perfil_step_214.step").val().scale(0.001)

# ── Geração dos pavimentos ───────────────────────────────────────────────────
lista_pav = []

for i in range(n_pav + 1):
    cota_atual = round(cota_inicial + pap * i, 2)
    scale_factor = np.abs(np.sin(i) * smooth_factor + 1) 

    perfil_escalado = cq.Workplane("XY").add(scale_xy(
                                                        perfil.val(), 
                                                        scale = [scale_factor,scale_factor,1],
                                                        translate=[0,0,cota_atual]
                                                        ))

    pav = (
        perfil_escalado
        .wires().toPending()
        .extrude(pap)
        .rotate((0, 0, 0), (0, 0, 1), i * rot_inc)
    )

    lista_pav.append(pav)

show(lista_pav)

### Exportação

In [23]:
# Create an assembly and add the objects
assy = cq.Assembly()
for i in range(len(lista_pav)):
    assy.add(lista_pav[i], name=f'pav_{i}')

# Export the assembly to a STEP file
assy.export(
    "output.step", 
    mode="default" # ou mode = "fused" para extrair os objetos unidos (solid union)
    )